In [90]:
%pip install "laspy[laszip]" pyproj

Note: you may need to restart the kernel to use updated packages.


In [91]:
import os
import sys

file_path = os.path.dirname(os.path.abspath(''))  # for .ipynb notebook
sys.path.append(file_path)

import numpy as np
import pyproj
import laspy
import torch

from src.data import Data
from src.utils.color import to_float_rgb

from pointai.pcd import convert_crs
from pointai.pcd.las import read_las_file

In [92]:
pcd = read_las_file('../data/here/HT068_1616462262_Mar2021_8q1rXkj7rMj.laz')

In [93]:
convert_crs(pcd, target_crs=pyproj.CRS.from_epsg(7855), in_place=True)

PointCloud(xyz=array([[3.25083745e+05, 5.81123162e+06, 1.82708292e+01],
       [3.25104420e+05, 5.81123149e+06, 1.58008292e+01],
       [3.25082922e+05, 5.81123190e+06, 1.88558292e+01],
       ...,
       [3.24898599e+05, 5.81117587e+06, 2.18608292e+01],
       [3.24916492e+05, 5.81113399e+06, 1.53983292e+01],
       [3.24899130e+05, 5.81117632e+06, 2.30033292e+01]]), rgb=array([[108, 116, 103],
       [138, 141, 150],
       [ 74,  77,  45],
       ...,
       [113, 106,  87],
       [146, 147, 156],
       [179, 174, 172]], dtype=uint8), crs=<Projected CRS: EPSG:7855>
Name: GDA2020 / MGA zone 55
Axis Info [cartesian]:
- E[east]: Easting (metre)
- N[north]: Northing (metre)
Area of Use:
- name: Australia - onshore and offshore between 144°E and 150°E.
- bounds: (144.0, -50.89, 150.01, -9.23)
Coordinate Operation:
- name: Map Grid of Australia zone 55
- method: Transverse Mercator
Datum: Geocentric Datum of Australia 2020
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich
, classificati

In [117]:
if pcd.intensity is not None and pcd.intensity.dtype == np.uint16:
    pcd.intensity = pcd.intensity.astype(np.float32) / 255.0
pcd.intensity

array([0.18431373, 0.03529412, 0.10196079, ..., 0.14509805, 0.03921569,
       0.12941177], dtype=float32)

In [120]:
data = Data()

pos = torch.from_numpy(pcd.xyz)
pos_offset = pos[0]
data.pos = (pos - pos_offset).to(torch.float32)
data.pos_offset = pos_offset

if pcd.rgb.dtype == np.uint8:
    pcd.rgb = pcd.rgb.astype(np.float32) / 255.0
data.rgb = to_float_rgb(torch.from_numpy(pcd.rgb))

if pcd.intensity is not None:
    data.intensity = torch.from_numpy(pcd.intensity)

In [95]:
# data.show(path='../data/here/my_demo.html')

In [97]:
from src.transforms import SampleRecursiveMainXYAxisTiling, GridSampling3D
from src.data import Batch

# Recursively tile the cloud into `2**pc_tiling` chunks with respect to 
# principal components of the XY coordiantes
pc_tiling = 3

# Voxelize the point cloud only for the sake of faster computation and 
# visualization here
# data_5m = GridSampling3D(5)(data)

# Compute each chunk 
chunks = []
for x in range(2**pc_tiling):
    # Extract the chunk at x in the recursive tiling
    # chunk = SampleRecursiveMainXYAxisTiling(x=x, steps=pc_tiling)(data_5m)
    chunk = SampleRecursiveMainXYAxisTiling(x=x, steps=pc_tiling)(data)

    # Add a 'tile' attribute to the points for visualization
    chunk.tile = torch.full((chunk.num_points,), x)
    
    # Store the chunk for later aggregation
    chunks.append(chunk)

# Aggregate all chunk `Data` objects into one big `Data` object
data_tiled = Batch.from_data_list(chunks)

In [98]:
from src.transforms import SampleXYTiling

# Extract the chunk at (x, y) in the tiling grid
data = SampleXYTiling(x=1, y=1, tiling=3)(data)

In [99]:
data.show(path='../data/here/my_demo.html')

In [122]:
from src.utils import init_config

cfg = init_config(overrides=[f"experiment=semantic/dales"])

In [123]:
cfg.keys()

dict_keys(['task_name', 'optimized_metric', 'tags', 'train', 'test', 'compile', 'ckpt_path', 'seed', 'float32_matmul_precision', 'datamodule', 'model', 'callbacks', 'logger', 'trainer', 'paths', 'extras'])

In [124]:
from src.transforms import instantiate_datamodule_transforms

transforms_dict = instantiate_datamodule_transforms(cfg.datamodule)
transforms_dict

{'pre_transform': Compose([
   SaveNodeIndex(key=sub),
   DataTo(device=cuda),
   GridSampling3D(grid_size=0.1, quantize_coords=False, mode=mean, bins={'y': 9}),
   KNN(k=25, r_max=10, oversample=False, self_is_neighbor=False),
   DataTo(device=cpu),
   PointFeatures(keys=('elevation', 'intensity', 'linearity', 'planarity', 'scattering', 'verticality'), k_min=1, k_step=-1, k_min_search=10, overwrite=True),
   GroundElevation(z_threshold=5, verticality_threshold=None, xy_grid=None, model=ransac, scale=20, kwargs={}),
   DataTo(device=cuda),
   AdjacencyGraph(k=10, w=1),
   ConnectIsolated(k=1),
   DataTo(device=cpu),
   AddKeysTo(keys=['linearity', 'planarity', 'scattering', 'elevation'], to=x, delete_after=False),
   CutPursuitPartition(regularization=[0.1, 0.2, 0.3], spatial_weight=[0.1, 0.01, 0.001], cutoff=[10, 30, 100], iterations=15, k_adjacency=10),
   NAGRemoveKeys(level=all, keys=('x',)),
   NAGTo(device=cuda),
   SegmentFeatures(n_max=128, n_min=32, keys=('log_length', 'log_si

In [125]:
# Apply pre-transforms
nag = transforms_dict['pre_transform'](data)

# Simulate the behavior of the dataset's I/O behavior with only
# `point_load_keys` and `segment_load_keys` loaded from disk
from src.transforms import NAGRemoveKeys
nag = NAGRemoveKeys(level=0, keys=[k for k in nag[0].keys if k not in cfg.datamodule.point_load_keys])(nag)
nag = NAGRemoveKeys(level='1+', keys=[k for k in nag[1].keys if k not in cfg.datamodule.segment_load_keys])(nag)

In [126]:
# Move to device
nag = nag.cuda()

# Apply on-device transforms
nag = transforms_dict['on_device_test_transform'](nag)

In [134]:
nag[0].keys

['semantic_pred', 'x', 'pos_offset', 'pos', 'super_index']

In [127]:
nag.show(keys=nag[0].keys, centroids=True, h_edge=True, path='../data/my_demo_transformed.html')

In [128]:
import hydra 
from src.utils import init_config

# Path to the checkpoint file downloaded from https://zenodo.org/records/8042712
ckpt_path = "../ckpt/spt-2_dales.ckpt"

cfg = init_config(overrides=[f"experiment=semantic/dales"])

# Instantiate the model and load pretrained weights
model = hydra.utils.instantiate(cfg.model)
model = model._load_from_checkpoint(ckpt_path)

Lightning automatically upgraded your loaded checkpoint from v1.8.4.post0 to v2.4.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../ckpt/spt-2_dales.ckpt`


In [129]:
nag

NAG(num_levels=4, num_points=[2141506, 31901, 7088, 2002], device=cuda:0)

In [130]:
# Set the model in inference mode on the same device as the input
model = model.eval().to(nag.device)

# Inference, returns a task-specific ouput object carrying predictions
with torch.no_grad():
    output = model(nag)

In [131]:
output.semantic_pred().shape, nag.num_points

(torch.Size([31901]), [2141506, 31901, 7088, 2002])

In [132]:
# Compute the level-0 (voxel-wise) semantic segmentation predictions 
# based on the predictions on level-1 superpoints and save those for 
# visualization in the level-0 Data under the 'semantic_pred' attribute
nag[0].semantic_pred = output.voxel_semantic_pred(super_index=nag[0].super_index)

In [133]:
from src.datasets.dales import CLASS_NAMES as DALES_CLASS_NAMES
from src.datasets.dales import CLASS_COLORS as DALES_CLASS_COLORS

nag.show(class_names=DALES_CLASS_NAMES, class_colors=DALES_CLASS_COLORS, path='../data/my_demo_pred.html')

In [89]:
# from src.datasets.kitti360 import CLASS_NAMES as KITTI360_CLASS_NAMES
# from src.datasets.kitti360 import CLASS_COLORS as KITTI360_CLASS_COLORS

# nag.show(class_names=KITTI360_CLASS_NAMES, class_colors=KITTI360_CLASS_COLORS, keys=nag[0].keys, path='../data/my_demo_pred.html')